[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/26_causal_attention.ipynb)

# 🟡 Medium: Causal Self-Attention

*Attention & Transformers*
Implement **causal (autoregressive) self-attention**: position $i$ may attend to
positions $0 \dots i$ and to nothing later.

$$\text{out} = \operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V,
\qquad
M_{ij} = \begin{cases} 0 & j \le i \\ -\infty & j > i \end{cases}$$

### Signature
`causal_attention(q, k, v)` with `q`, `k`, `v` all `(..., T, d)` and the same `T`
(this is self-attention). Returns `(..., T, d)`. Leading axes are free, so
`(B, T, D)` and `(B, H, T, D_h)` must both work.

### Rules
- No `jax.nn.dot_product_attention`, no `nnx.MultiHeadAttention`, no
  `is_causal=` shortcut from a library
- Build the mask with `jnp.tril` / `jnp.triu` — no Python loop over positions
- The mask is applied to the **scores**, before `jax.nn.softmax`
- Must be jittable and differentiable

### Add before, do not multiply after
The tempting wrong version is:

```python
w = jax.nn.softmax(scores, axis=-1)
w = w * causal          # WRONG
out = w @ v
```

This does not merely zero the future — it corrupts the past. The softmax
denominator was computed over **all** $T$ positions, including the ones you then
deleted, so row $i$ now sums to $\sum_{j\le i} p_{ij} < 1$ instead of $1$. Two
consequences:

1. **Magnitude collapse.** Row $0$ keeps only $p_{00}$, typically $\approx 1/T$,
   so the first token's output is shrunk by ~$T\times$ while the last token's is
   untouched. The layer applies a position-dependent gain that LayerNorm then
   has to undo.
2. **Information leak.** The denominator is a function of the future keys. Even
   with the weights zeroed, $\partial\,\text{out}_0 / \partial k_5 \ne 0$, so a
   language model trained this way is reading its own labels. It will show an
   impossibly low training loss and generate garbage at inference, because at
   decode time the future keys do not exist.

Adding $-\infty$ (or a large negative number) *before* the softmax makes the
blocked logits contribute exactly $0$ to the denominator, so each row is a proper
distribution over its visible prefix and the gradient w.r.t. future keys is
exactly zero.

### Which large negative number
`-1e9` is the usual choice and is fine in `float32`/`bfloat16`. In `float16` it
overflows to `-inf` — harmless here, but if you ever add two such biases (causal
+ padding) you get `-inf + -inf`, and a fully-masked row then produces `nan`
because `softmax` computes `x - max(x) = -inf - (-inf)`. `jnp.finfo(dtype).min`
or `jnp.where(mask, scores, -1e9)` (replace rather than add) are the defensive
forms. Padding-only rows in a batch are the real-world case that hits this.

### At decode time
With a KV cache, $Q$ has length $1$ while $K$ has length $t+1$ — the mask becomes
all-ones and disappears. If you ever see a `tril` applied to a non-square score
matrix during generation, the alignment is wrong: causality is about absolute
positions, not about the shape of the block you happen to be computing.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def causal_attention(q, k, v):
    """Scaled dot-product self-attention with future positions masked out.

    Args:
        q: (..., T, d)
        k: (..., T, d)
        v: (..., T, d)

    Returns:
        (..., T, d) — position i attends only to positions 0..i.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

T, D = 5, 4
q = jax.random.normal(jax.random.key(0), (1, T, D))
k = jax.random.normal(jax.random.key(1), (1, T, D))
v = jnp.arange(T * D, dtype=jnp.float32).reshape(1, T, D)

# Look at the weight matrix the mask produces.
scores = (q @ jnp.swapaxes(k, -1, -2)) / jnp.sqrt(float(D))
allowed = jnp.tril(jnp.ones((T, T), dtype=bool))
w_right = jax.nn.softmax(scores + jnp.where(allowed, 0.0, -1e9), axis=-1)
w_wrong = jax.nn.softmax(scores, axis=-1) * allowed

print("row sums, mask BEFORE softmax:", w_right.sum(-1)[0])
print("row sums, mask AFTER  softmax:", w_wrong.sum(-1)[0])
print("-> the 'after' version shrinks early positions toward zero")

out = causal_attention(q, k, v)
print("position 0 output:", out[0, 0], " == v[0]:", v[0, 0])

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("causal_attention")

# hint("causal_attention")      # stuck? nudge without the answer
# solution("causal_attention")  # spoiler: the reference implementation